In [1]:
1

1

In [2]:
import os, getpass

os.environ.setdefault("DEEPEVAL_TELEMETRY_OPT_OUT", "YES")  # skip DeepEval's anonymous telemetry

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ_API_KEY: ")

print("Key set.")

Key set.


## Test Case

In [3]:
from deepeval.test_case import LLMTestCase

test_cases = [
    LLMTestCase(
        input="What's your refund policy?",
        actual_output=(
            "Happy to help! We offer full refunds within 30 days of purchase, no questions "
            "asked. Just reply here with your order number."
        ),
        expected_output="Refunds are available within 30 days of purchase.",
    ),
    LLMTestCase(
        input="What's your refund policy?",
        actual_output="No refunds. Read the policy page next time.",
        expected_output="Refunds are available within 30 days of purchase.",
    ),
]

## The judge: `LocalModel` (Groq)

In [4]:
from deepeval.models import LocalModel

judge = LocalModel(
    model="llama-3.3-70b-versatile",
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
    temperature=0,
)
print("Judge ready:", judge.get_model_name())

Judge ready: llama-3.3-70b-versatile (Local Model)


## Define two G-Eval metrics

In [5]:
from deepeval.metrics import GEval
from deepeval.test_case import SingleTurnParams

# correctness = GEval(
#     name="Correctness",
#     criteria="Determine whether the actual output is factually correct given the expected output.",
#     evaluation_params=[
#         SingleTurnParams.INPUT,
#         SingleTurnParams.ACTUAL_OUTPUT,
#         SingleTurnParams.EXPECTED_OUTPUT,
#     ],
#     model=judge,
# )

tone = GEval(
    name="Professional Tone",
    evaluation_steps=[
        "Check whether the response is polite and professional.",
        "Penalize sarcasm, rudeness, or dismissive language.",
        "Reward clear, respectful phrasing even if brief.",
    ],
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT],
    model=judge,
)

In [6]:
# from deepeval.metrics.g_eval import Rubric, GEval
# from deepeval.test_case import SingleTurnParams

# correctness = GEval(
#     name="Correctness",
#     criteria="Determine whether the actual output is factually correct given the expected output.",
#     evaluation_params=[
#         SingleTurnParams.INPUT,
#         SingleTurnParams.ACTUAL_OUTPUT,
#         SingleTurnParams.EXPECTED_OUTPUT,
#     ],
#     rubric=[
#         Rubric(score_range=(0, 2), expected_outcome="Factually wrong or contradicts the expected output."),
#         Rubric(score_range=(3, 5), expected_outcome="Partially correct -- misses or garbles a key fact."),
#         Rubric(score_range=(6, 8), expected_outcome="Mostly correct with only minor omissions."),
#         Rubric(score_range=(9, 10), expected_outcome="Fully correct and complete."),
#     ],
#     model=judge,  # your existing LocalModel(Groq) judge
# )

## Run the evaluation

In [7]:
from deepeval import evaluate
from deepeval.evaluate.configs import AsyncConfig, DisplayConfig, ErrorConfig

results = evaluate(
    test_cases=test_cases,
    # metrics=[correctness, tone],
    metrics=[tone],
    async_config=AsyncConfig(max_concurrent=2),           # be gentle on Groq's rate limits
    display_config=DisplayConfig(print_results=True),
    error_config=ErrorConfig(ignore_errors=True, skip_on_missing_params=True),
)

✨ You're running DeepEval's latest Professional Tone [GEval] Metric! (using llama-3.3-70b-versatile (Local Model),
strict=False, async_mode=True)...

/Users/eshantdas/Desktop/SelfStudy/PersonalTest/KrishNaikUdemyLLMSecurity_gateways/.venv/lib/python3.13/site-packag
es/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:              What's your refund policy?                                                           │
│  │     Actual Output:      No refunds. Read the policy page next time.                                          │
│  │     Expected Output:    Refunds are available within 30 days of purchase.                                    │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric                    ┃ Score ┃ Threshold ┃ Reason                                           │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Professional Tone [GEval] │ 0.20  │ 0.50      │ The response is brief but lacks politeness and   │
│              │                           │       │           │ professionalism, using dismissive language by    │
│              │                           │       │           │ saying 'next time', which is penalized           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                              ┃ Average Score       ┃ Pass Rate                               ┃ Total    │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━ │
│  Professional Tone [GEval]           │ 0.60                │ 50.00% | passed=1 | failed=1            │ 2        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=6487349;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 1.41s | token cost: None)
» Test Results (2 total tests):
   » Pass Rate: 50.0% | Passed: 1 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [9]:
results.test_results

[TestResult(name='test_case_1', success=False, metrics_data=[MetricData(name='Professional Tone [GEval]', threshold=0.5, success=False, score=0.2, reason="The response is brief but lacks politeness and professionalism, using dismissive language by saying 'next time', which is penalized", strict_mode=False, flaky=False, evaluation_model='llama-3.3-70b-versatile (Local Model)', error=None, evaluation_cost=0.0, input_tokens=0, output_tokens=0, verbose_logs='Criteria:\nNone \n \nEvaluation Steps:\n[\n    "Check whether the response is polite and professional.",\n    "Penalize sarcasm, rudeness, or dismissive language.",\n    "Reward clear, respectful phrasing even if brief."\n] \n \nRubric:\nNone \n \nScore: 0.2')], conversational=False, index=1, multimodal=False, input="What's your refund policy?", actual_output='No refunds. Read the policy page next time.', expected_output='Refunds are available within 30 days of purchase.', context=None, retrieval_context=None, turns=None, metadata=None

## CHecl excalDraw for Best Use Cases and Limitations